# Part 5: The Analyst Report

After you have successfully deployed your pipeline and run the **Burst** profile (500 messages) in the test apparatus, you need to extract the results and answer a few questions.

We use `boto3` to scan the DynamoDB table, handling pagination automatically, and convert the results into standard Python dictionaries and floats.

## Setup: Configure Your Student ID
Replace `YOURID` below with the exact student ID you used for deployment.

In [12]:
%pip install boto3
STUDENT_ID = "muhammedaltindal"  # <--- Change this
TABLE_NAME = f"adflow-{STUDENT_ID}-results"
REGION = "us-east-1"
print(f"Target Table: {TABLE_NAME}")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Target Table: adflow-muhammedaltindal-results


## Step 1: Export Data from DynamoDB
This cell connects to your DynamoDB table, downloads all records, and converts the Decimal values back to standard floats.

In [13]:
import boto3
from decimal import Decimal
from collections import Counter

# Note: This uses your active AWS credentials (from `aws configure` or exported environment variables)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(TABLE_NAME)

results = []
response = table.scan()
results.extend(response.get("Items", []))

# Handle pagination if the table has more than 1 MB of data
while "LastEvaluatedKey" in response:
    response = table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
    results.extend(response.get("Items", []))

print(f"\nLoaded {len(results)} records from DynamoDB.")

# Convert Decimal types to Python floats for easier math/plotting
for item in results:
    for key in ["winning_bid_amount", "winning_score", "score_margin"]:
        if key in item and isinstance(item[key], Decimal):
            item[key] = float(item[key])

if results:
    print("\nSample record:")
    print(results[0])


Loaded 500 records from DynamoDB.

Sample record:
{'processed_at': '2026-03-18T19:06:03.927528+00:00', 'opportunity_id': '1da91895-9179-40f8-9170-389f05cdddd1', 'winning_score': 5.247, 'score_margin': 1.0713999999999997, 'winning_advertiser_id': 'adv_fintech_02', 'winning_bid_amount': 4.77, 'content_category': 'sports'}


## Section 1: Pipeline Evidence
Print the total records and a quick count of auction wins per advertiser across the entire dataset to prove your pipeline successfully routed messages.

In [18]:
# TODO: Print the total number of records
print(f"Total pipeline records: {len(results)}")

# TODO: Compute and print the auction wins per advertiser (overall)
# Hint: Use collections.Counter on the 'winning_advertiser_id' field

# Total records
print(f"Total pipeline records: {len(results)}")

# Auction wins per advertiser
from collections import Counter

win_counts = Counter(item["winning_advertiser_id"] for item in results)

print("\nAuction wins per advertiser:")
for adv, count in win_counts.items():
    print(f"{adv}: {count}")


Total pipeline records: 500
Total pipeline records: 500

Auction wins per advertiser:
adv_fintech_02: 16
adv_insurance_01: 43
adv_telecom_01: 23
adv_sportswear_02: 4
adv_auto_02: 33
adv_energy_01: 27
adv_streaming_01: 37
adv_fastfood_01: 47
adv_beauty_01: 14
adv_auto_01: 65
adv_fintech_01: 75
adv_sportswear_01: 18
adv_travel_01: 54
adv_fastfood_02: 21
adv_beauty_02: 3
adv_insurance_02: 6
adv_travel_02: 3
adv_gaming_01: 6
adv_energy_02: 4
adv_streaming_02: 1


**Evidence Requirement:** Don't forget to push a screenshot of the **Test Apparatus** (showing a completed Burst run) to a `screenshots/` directory in this repo when submitting.

---
## Q1: Results Analysis

**Question:** Which advertiser won the most auctions overall? Which advertiser won the most in the `sports` content category specifically? Why do the overall and sports-specific rankings differ? Explain in 2–3 sentences, referencing the relevance multiplier table.

In [20]:
# TODO: Find the top winner in the 'sports' category
sports_results = [r for r in results if r.get("content_category") == "sports"]
print(f"Sports records: {len(sports_results)}")

from collections import Counter

# Overall top winner
overall_counts = Counter(r["winning_advertiser_id"] for r in results)
overall_top, overall_top_count = overall_counts.most_common(1)[0]

# Sports-only top winner
sports_results = [r for r in results if r.get("content_category") == "sports"]
sports_counts = Counter(r["winning_advertiser_id"] for r in sports_results)
sports_top, sports_top_count = sports_counts.most_common(1)[0]

print("Overall top winner:", overall_top, overall_top_count)
print("Sports top winner:", sports_top, sports_top_count)

Sports records: 128
Overall top winner: adv_fintech_01 75
Sports top winner: adv_auto_01 20


**Your Answer (Q1):**

The advertiser that won the most auctions overall was adv_fintech_01 with 75 wins. In the sports content category, the top advertiser was adv_auto_01 with 20 wins. The rankings differ because the scoring system applies a relevance multiplier based on how well an advertiser’s category matches the content category. In sports-related content, advertisers that are more contextually relevant (e.g., auto or sports-related categories) receive a higher multiplier, allowing them to outperform others even if their raw bids are lower.

---
## Q2: Code Reflection

Answer **one** of the following (your choice):
 
* **Option A (Scale & Limits):** The test apparatus sent messages in small batches. If traffic suddenly spiked from 10 opportunities a second to 10,000 a second, what specific components of our current pipeline (SQS limits, Lambda concurrency, DynamoDB throughput) would become bottlenecks first, and what AWS settings would you adjust to handle the load?
* **Option B (The Distributed Process):** Writing code for an event-driven, queue-based pipeline is very different from writing a single local script. What was the most challenging part of getting SQS, Lambda, and DynamoDB to communicate correctly, or the most confusing bug you encountered, and what did it teach you about distributed architecture?

A well-argued two-paragraph response is sufficient for either option.

**Your Answer (Q2):**

Option B

The most challenging part of this project was making sure that SQS, Lambda, and DynamoDB were all connected correctly and using the exact same naming convention. A small mismatch in the student ID or resource name caused the test apparatus to fail even when the Lambda itself was deployed. Another confusing issue was that the results were appearing in the results queue, but DynamoDB was empty after cleanup, which made it clear that in distributed systems, one component can appear healthy while another part of the pipeline is failing or misconfigured. This taught me that debugging event-driven systems requires checking each service independently rather than assuming the whole pipeline is working.

I also learned that distributed architecture is much more sensitive to configuration details than a single local Python script. In a local script, data moves through one process and errors are usually immediate and visible. In this project, the workflow depended on AWS permissions, queue URLs, event source mappings, environment variables, and consistent resource names. That experience showed me that building distributed systems is not only about writing correct logic, but also about carefully managing infrastructure, permissions, and communication between services.